In [ ]:
import scipp as sc
import plopp as pp
import scippneutron as scn
import scippnexus as snx
import h5py
from pathlib import Path
import numpy as np

## A simulated sample with incoherent elastic scattering and one phonon mode
The data file specified below contains simulated scattering from a sample with one incoherent elastic scattering mode,
and one phonon mode with instrument-parameter controlled slope, `sound_speed`.

A standard-McStas particle can only scatter from either the elastic mode _or_ phonon,
and this simulation splits the particles equally between the two modes.
A better approach would have been sending significantly more particles to the phonon.

The phonon FCC lattice constant $a=6.56162$ Å was chosen such that its {200} Bragg peaks (not simulated) 
would appear at 90 degree $a_4$ in the $E_f = 3.8$ meV analyzers.

The simulation was conducted as an sample orientation scan with all other parameters fixed.

| Parameter | Value | Notes |
|-----------|-------|-------|
| `sound_speed` / meV Å | 2.5 |  |
| $a_3$ / degree | 0:179 | 1-degree steps, endpoints included |
| $a_4$ / degree | 90. | all simulations have this single detector tank position |
| pulse-shaping chopper opening time  / msec | 0.2 | picked to be a realistic best-case energy-resolution |
| minimum $E_i$ / meV | 2.5 | giving a maximum of ~5.1 meV due to BIFROST's pseudo-white beam |

The simulation was started at approxmately 5:37 UTC on 14. September 2024, 
and required approximately 30 seconds per $a_3$ setting using MPI with 6 nodes.

In [ ]:
datafile = "20240914/BIFROST_20240914T053723.h5"

In [ ]:
from bifrost2409.config import POOCH_DATA_DIR, INTERIM_DATA_DIR
from bifrost2409.dataset import download_datafiles
download_datafiles([datafile])

### Trust the workflow, use the worflow

Now that we trust the process that we performed 'by hand' in the last two notebooks,
we can make use of the same process available through the workflow in abbreviated form
as `bifrost_single`, which _ignores_ any scan information and treats all data as a single setting.

> **Note**: Since the workflow is _intended_ for _real_ data, but our simulations need some manipulations
> you _must_ specify that this datafile `is_simulated`.

In [ ]:
from ess import bifrost
from ess.bifrost.data import tof_lookup_table_simulation
from ess.spectroscopy.types import *

In [ ]:
with snx.File(POOCH_DATA_DIR / datafile) as f:
    detector_names = list(f['entry/instrument'][snx.NXdetector])

In [ ]:
workflow = bifrost.BifrostSimulationWorkflow(detector_names)
workflow[Filename[SampleRun]] = POOCH_DATA_DIR / datafile
workflow[TimeOfFlightLookupTable] = sc.io.load_hdf5(tof_lookup_table_simulation())
workflow[PreopenNeXusFile] = PreopenNeXusFile(True)

In [ ]:
workflow.visualize(EnergyData[SampleRun], graph_attr={"rankdir": "LR"}, compact=True)

In [ ]:
scheduler = sciline.scheduler.NaiveScheduler()
data = workflow.compute(EnergyData[SampleRun], scheduler=scheduler)

We can get an overview of the per-pixel inelastic spectrum (but we plot it here as a function of 10% of a _tube_)

In [ ]:
data.bins.concat(['a3', 'a4']).hist(
    energy_transfer=sc.linspace('energy_transfer', -1.7, 1.7, 50, unit='meV'),
    detector_number=sc.index(10),
).plot(norm='log')

Converting this data to S(**Q**, E) requires splitting the continuous measurement-time dimension of the data into discrete 
periods with constant settings.
> **Note**: currently, this can be done _exactly_ since the `NXlog` values of simulated parameters are stable and precise.
> The same should be true in most cases for parameter _set points_, so they will be used to segment real data.
>

A different workflow entry-point `bifrost` segments the data (currently limited to constant ($a_3$, $a_4$) pairs),
then calls, effectively, `bifrost_single` per segment before calculating **Q** _in the sample table coordinate system_ 
and combining the data.

_This process is unoptimized and slow -- about 15 minutes on my laptop, so we will store the result to avoid re-creating it unnecessarily_

In [ ]:
q = data.bins.coords.pop('sample_table_momentum_transfer')
data.bins.coords['table_momentum_x'] = q.fields.x.copy()
data.bins.coords['table_momentum_z'] = q.fields.z.copy() # TODO

In [ ]:
def hist_q_e(events, qx, qz, e):
    return events.bins.concat().hist(
        energy_transfer=e,
        table_momentum_x=qx,
        table_momentum_z=qz,
    ).squeeze()

In [ ]:
hist_q_e(
    data,
    qx=200,
    qz=200,
    e=sc.array(values=[-0.05, 0.05], dims=['energy_transfer'], unit='meV'),
).plot(norm='log')

In [ ]:
hist_q_e(
    data,
    qx=200,
    qz=200,
    e=sc.array(values=[0.3, 0.9], dims=['energy_transfer'], unit='meV'),
).plot(norm='log')

In [ ]:
hist_q_e(
    data,
    qx=200,
    qz=200,
    e=sc.array(values=[1., 1.6], dims=['energy_transfer'], unit='meV'),
).plot(norm='log')

In [ ]:
astar = 2 * np.pi / 6.56162

In [ ]:
hist_q_e(
    data,
    qx=sc.array(values=[-2 * astar - 0.2,  -2 * astar + 0.2], dims=['table_momentum_x'], unit='1/angstrom'),
    qz=200,
    e=50,
).plot(norm='log')